# XGBoost + Optuna

Tune with five stratified folds on the training split. Use the 20% holdout only for final evaluation. The submission uses default probabilities unless `SUBMISSION_MODE` is set to `"labels"`.

In [1]:
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from xgboost import XGBClassifier

PROJECT_DIR = Path(r"C:\Users\Admin\Desktop\coding\data sci\aihack")
TARGET = "12-Month Default Flag"
N_TRIALS = 30
SUBMISSION_MODE = "probabilities"  # Set to "labels" only if the rules require 0/1.
OUTPUT_PATH = PROJECT_DIR / "submission" / "submission_V2.csv"

df = pd.read_csv(PROJECT_DIR / "data_preprocessed_V1.csv", dtype={"ID": "string"})
df_test = pd.read_csv(PROJECT_DIR / "test_preprocessed_V1.csv", dtype={"ID": "string"})
if len(df_test) != 137_091 or TARGET in df_test.columns:
    raise ValueError("Expected the preprocessed competition test with 137,091 rows and no target.")

In [2]:
X = df[['Is_male', 'Is_married',
       'Declared Number of Borrowings',
       'Declared Borrowing Amount', 'Rent Burden',
       'Number of Dependents', 'Number of Dependent Children',
       'Home Phone Number Available', 'Mobile Phone Number Available',
       'Employer Phone Number Available', 'Driver Licence Number Available',
       'Log Declared Annual Income', 'Log Rent Burden',
       'Log Desired Credit Limit', 'Has_rent', 'Age_bins_25–34',
       'Age_bins_35–44', 'Age_bins_45–54', 'Age_bins_55–64', 'Age_bins_65+',
       'Application Channel_17', 'Application Channel_18',
       'Application Channel_20', 'Business Entity Type_1',
       'Business Entity Type_2', 'Business Entity Type_5',
       'Business Entity Type_6', 'Business Entity Type_7',
       'Business Entity Type_9', 'Company Size Category_2',
       'Company Size Category_3', 'Company Size Category_4',
       'Company Size Category_5', 'Company Size Category_6',
       'Company Size Category_7', 'Company Size Category_8',
       'Company Size Category_9', 'Employment Type Category_2',
       'Employment Type Category_3', 'Employment Type Category_4',
       'Employment Type Category_5', 'Employment Type Category_6',
       'Employment Type Category_7', 'Employment Type Category_8',
       'Industry Category_2', 'Industry Category_3', 'Industry Category_4',
       'Industry Category_5', 'Industry Category_6', 'Industry Category_7',
       'Industry Category_8', 'Industry Category_9', 'Industry Category_10',
       'Industry Category_11', 'Industry Category_12', 'Industry Category_13',
       'Industry Category_15', 'Industry Category_16', 'Industry Category_17',
       'Industry Category_18', 'Industry Category_99', 'Insurance Type_1',
       'Insurance Type_3', 'Insurance Type_4', 'Occupation Category_240',
       'Occupation Category_340', 'Occupation Category_440',
       'Occupation Category_540', 'Occupation Category_640',
       'Occupation Category_740', 'Occupation Category_840',
       'Occupation Category_940', 'Occupation Category_1040',
       'Occupation Category_1140', 'Occupation Category_1240',
       'Occupation Category_1340', 'Occupation Category_1440',
       'Occupation Category_1540', 'Payday_Period_Mid_month',
       'Payday_Period_Late_month', 'Residence Ownership Status_1',
       'Residence Ownership Status_2', 'Residence Ownership Status_3',
       'Residence Type_2', 'Residence Type_3', 'Residence Type_4',
       'Residence Type_5', 'Residence Type_6', 'Residence_age_bins_1–5 years',
       'Residence_age_bins_5–10 years', 'Residence_age_bins_10–20 years',
       'Residence_age_bins_20-40 years', 'Residence_age_bins_40-60 years',
       'Residence_age_bins_60+ years']]
X = X.astype("float32")
y = df[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=67,
)
print(f"Training positive rate: {y_train.mean():.1%}")

Training positive rate: 9.7%


In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
class_ratio = (y_train == 0).sum() / (y_train == 1).sum()

fixed_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": 4,
}


def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=50),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 30.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, max(2.0, class_ratio), log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 30.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }
    model = XGBClassifier(**fixed_params, **params)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", error_score="raise")
    return scores.mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.enqueue_trial({
    "learning_rate": 0.05, "n_estimators": 300, "max_depth": 4,
    "min_child_weight": 5.0, "subsample": 0.85, "colsample_bytree": 0.85,
    "scale_pos_weight": 1.0, "reg_alpha": 0.0, "reg_lambda": 1.0, "gamma": 0.0,
})
study.optimize(objective, n_trials=N_TRIALS)

best_model = XGBClassifier(**fixed_params, **study.best_params)
best_model.fit(X_train, y_train)
default_idx = list(best_model.classes_).index(1)

print("Best parameters:", study.best_params)
print(f"Mean CV ROC-AUC: {study.best_value:.4f}")

[I 2026-11-10 02:30:01,545] A new study created in memory with name: no-name-1a924a50-25c4-4d69-abb6-ca39fefec368
[I 2026-11-10 02:30:04,927] Trial 0 finished with value: 0.6395452828508725 and parameters: {'learning_rate': 0.05, 'n_estimators': 300, 'max_depth': 4, 'min_child_weight': 5.0, 'subsample': 0.85, 'colsample_bytree': 0.85, 'scale_pos_weight': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'gamma': 0.0}. Best is trial 0 with value: 0.6395452828508725.
[I 2026-11-10 02:30:12,591] Trial 1 finished with value: 0.6386410876827788 and parameters: {'learning_rate': 0.030710573677773714, 'n_estimators': 800, 'max_depth': 6, 'min_child_weight': 7.6611007077713635, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'scale_pos_weight': 1.1385365444083382, 'reg_alpha': 8.661761457749352, 'reg_lambda': 3.083434817935579, 'gamma': 3.540362888980227}. Best is trial 0 with value: 0.6395452828508725.
[I 2026-11-10 02:30:22,854] Trial 2 finished with value: 0.6396936462684162

Best parameters: {'learning_rate': 0.08095749207359969, 'n_estimators': 300, 'max_depth': 3, 'min_child_weight': 4.3035247290264556, 'subsample': 0.8251867852889606, 'colsample_bytree': 0.9940534359920857, 'scale_pos_weight': 3.5111228325425023, 'reg_alpha': 7.2255299993740865, 'reg_lambda': 0.7638524719259594, 'gamma': 0.8235022436976027}
Mean CV ROC-AUC: 0.6405


In [4]:
# Choose a binary threshold on out-of-fold training scores only.
oof_prob = cross_val_predict(
    best_model, X_train, y_train, cv=cv, method="predict_proba",
)[:, default_idx]
precision, recall, thresholds = precision_recall_curve(y_train, oof_prob)
p, r = precision[:-1], recall[:-1]
denominator = 4 * p + r
f2 = np.divide(5 * p * r, denominator, out=np.zeros_like(p), where=denominator > 0)
threshold = thresholds[np.argmax(f2)]

y_prob = best_model.predict_proba(X_test)[:, default_idx]
y_pred = (y_prob >= threshold).astype(int)
print(f"F2 threshold:           {threshold:.4f}")
print(f"Holdout ROC-AUC:        {roc_auc_score(y_test, y_prob):.4f}")
print(f"Holdout Average Precision: {average_precision_score(y_test, y_prob):.4f}")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
print(classification_report(y_test, y_pred, zero_division=0))

F2 threshold:           0.2100
Holdout ROC-AUC:        0.6423
Holdout Average Precision: 0.1499
[[4411 7972]
 [ 240 1087]]
              precision    recall  f1-score   support

           0       0.95      0.36      0.52     12383
           1       0.12      0.82      0.21      1327

    accuracy                           0.40     13710
   macro avg       0.53      0.59      0.36     13710
weighted avg       0.87      0.40      0.49     13710



In [5]:
gain = best_model.get_booster().get_score(importance_type="gain")
feature_importance = pd.Series(gain, name="Mean split gain").reindex(X.columns, fill_value=0)
display(feature_importance.sort_values(ascending=False).head(20).to_frame())

,Mean split gain
Residence Type_4,78.391022
Home Phone Number Available,49.346218
Is_married,42.604683
Application Channel_17,37.577293
Declared Number of Borrowings,37.159824
Driver Licence Number Available,33.473782
Employment Type Category_2,31.642630
Insurance Type_3,28.654837
Residence Ownership Status_3,27.470562
Company Size Category_7,26.664217


In [6]:
if SUBMISSION_MODE not in {"probabilities", "labels"}:
    raise ValueError('SUBMISSION_MODE must be "probabilities" or "labels".')

X_submit = df_test.loc[:, X.columns].astype("float32")
probabilities = best_model.predict_proba(X_submit)[:, default_idx]
submission_values = (
    (probabilities >= threshold).astype(int)
    if SUBMISSION_MODE == "labels"
    else probabilities
)
ids = df_test["ID"].str.strip().str.replace(r"\.0+$", "", regex=True)
if ids.isna().any() or ids.eq("").any() or not ids.is_unique:
    raise ValueError("The competition test contains missing or duplicate IDs.")

submission = pd.DataFrame({"ID": ids, TARGET: submission_values})
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Saved {len(submission):,} rows to {OUTPUT_PATH}")

Saved 137,091 rows to C:\Users\Admin\Desktop\coding\data sci\aihack\submission\submission_V2.csv
